In [1]:
import numpy as np 
import re
from scipy.interpolate import CubicSpline
from astropy.table import Table
import astropy.units as u
from astropy.cosmology import FlatLambdaCDM
from astropy.constants import c
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt

In [2]:
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

In [3]:
#Read all the files from the SED modeling. 
folder = "../../SED_Modeling_LegacySurvey"

#Band information
bands = Table.read("{}/bandmag.dat".format(folder), format='ascii')
for i, hn in enumerate(['bname', 'bcal', 'jyzero']):
    bands.rename_column("col{}".format(i+1), hn)
bands['bname'][-2] = "g_misc"
bands['bname'][-1] = "r_misc"

#SED model parameters
d20 = Table.read("{}/double.20".format(folder), format='ascii')
for i, hn in enumerate(['i', 'redshift', 'chi2', 'vec1', 'vec2', 'vec3', 'vec4', 'vec5']):
    d20.rename_column("col{}".format(i+1), hn)
d22 = Table.read("{}/double.22".format(folder), format='ascii')
for i, hn in enumerate(['i', 'ebv1', 'ebv2', 'igm', 'wid']):
    d22.rename_column("col{}".format(i+1), hn)

#The Pran probs
pran = dict()
cat = open("{}/f_stat/comp.sort".format(folder))
for line in cat:
    x = line.split()
    pran[x[0]] = float(x[-1])

In [4]:
#Get the errors. 
mc_d20 = Table.read("{}/MC/double.20".format(folder), format='ascii')
for i, hn in enumerate(['i', 'redshift', 'chi2', 'vec1', 'vec2', 'vec3', 'vec4', 'vec5']):
    mc_d20.rename_column("col{}".format(i+1), hn)

mc_d22 = Table.read("{}/MC/double.22".format(folder), format='ascii')
for i, hn in enumerate(['i', 'ebv1', 'ebv2', 'igm']):
    mc_d22.rename_column("col{}".format(i+1), hn)

nrep = int(len(mc_d20)/len(d20))
probs = np.array([68.3, 95.4, 99.7])
intervals = np.concatenate([50-probs/2., 50+probs/2.])
for col in ['vec1', 'vec5']:
    for j in range(3):
        d20[col+'_l{}'.format(j+1)] = 0.
        d20[col+'_u{}'.format(j+1)] = 0.
    for i in range(len(d20)):
        aux = np.percentile(mc_d20[col][i*nrep:(i+1)*nrep], intervals)
        for j in range(3):
            d20[col+'_l{}'.format(j+1)][i] = d20[col][i] - aux[j]
            d20[col+'_u{}'.format(j+1)][i] = aux[j+3] - d20[col][i]

for col in ['ebv1', 'ebv2']:
    for j in range(3):
        d22[col+'_l{}'.format(j+1)] = 0.
        d22[col+'_u{}'.format(j+1)] = 0.
    for i in range(len(d22)):
        aux = np.percentile(mc_d22[col][i*nrep:(i+1)*nrep], intervals)
        for j in range(3):
            d22[col+'_l{}'.format(j+1)][i] = d22[col][i] - aux[j]
            d22[col+'_u{}'.format(j+1)][i] = aux[j+3] - d22[col][i]


# for col in ['evec1_u', 'evec1_l', 'evec5_u', 'evec5_l']:
#     d20[col] = 0.0
# for col in ['ebv1_u', 'ebv1_l', 'ebv2_u', 'ebv2_l']:
#     d22[col] = 0.0

# nrep = int(len(mc_d20)/len(d20))
# print(nrep)
# for i in range(len(d20)):
#     d20['evec1_u'][i] = np.percentile(mc_d20['vec1'][i*nrep:(i+1)*nrep], 50.+68.3/2)
#     d20['evec1_l'][i] = np.percentile(mc_d20['vec1'][i*nrep:(i+1)*nrep], 50.-68.3/2)
#     d20['evec5_u'][i] = np.percentile(mc_d20['vec5'][i*nrep:(i+1)*nrep], 50.+68.3/2)
#     d20['evec5_l'][i] = np.percentile(mc_d20['vec5'][i*nrep:(i+1)*nrep], 50.-68.3/2)
#     d22['ebv1_u'][i]  = np.percentile(mc_d22['ebv1'][i*nrep:(i+1)*nrep], 50.+68.3/2)
#     d22['ebv1_l'][i]  = np.percentile(mc_d22['ebv1'][i*nrep:(i+1)*nrep], 50.-68.3/2)
#     d22['ebv2_u'][i]  = np.percentile(mc_d22['ebv2'][i*nrep:(i+1)*nrep], 50.+68.3/2)
#     d22['ebv2_l'][i]  = np.percentile(mc_d22['ebv2'][i*nrep:(i+1)*nrep], 50.-68.3/2)

# d20['evec1_u'] = d20['evec1_u'] - d20['vec1']
# d20['evec1_l'] = d20['vec1'] - d20['evec1_l']
# d20['evec5_u'] = d20['evec5_u'] - d20['vec5']
# d20['evec5_l'] = d20['vec5'] - d20['evec5_l']
# d22['ebv1_u']  = d22['ebv1_u'] - d22['ebv1']
# d22['ebv1_l']  = d22['ebv1'] - d22['ebv1_l']
# d22['ebv2_u']  = d22['ebv2_u'] - d22['ebv2']
# d22['ebv2_l']  = d22['ebv2'] - d22['ebv2_l']


In [5]:
d22.show_in_notebook()

idx,i,ebv1,ebv2,igm,wid,ebv1_l1,ebv1_u1,ebv1_l2,ebv1_u2,ebv1_l3,ebv1_u3,ebv2_l1,ebv2_u1,ebv2_l2,ebv2_u2,ebv2_l3,ebv2_u3
0,1,5.011872,0.03162278,1.4,W0019-1046,0.3221457944999999,0.4566297979999998,0.6405640350000006,0.9581457169999998,1.0308000000000002,1.297701,0.03162278,0.03147295,0.03162278,0.06837722,0.03162278,0.06837722
1,2,4.253576,0.0,0.4567042,W0116-0505,0.7456801134999997,0.9986133680000009,1.1274484139999998,3.579420444000001,1.5701578059999997,4.845881609499965,0.0,0.01,0.0,0.01995262,0.0,0.03162278
2,3,9.690195,0.1,1.4,W0204-0506,1.7469129999999993,1.7761171900000008,2.7960706559999995,4.4042100699999995,3.4079503769999997,7.501086569999991,0.0,0.0,0.0,0.0,0.020567180000000004,0.0
3,4,7.56583,0.0,0.0,W0220+0137,1.764893538,1.9283686090000014,3.1144825030000005,3.9973169700000035,3.8130284010000013,5.890717124999968,0.0,0.0,0.0,0.0,0.0,0.01
4,5,3.162278,0.03162278,1.4,W0831+0140,0.6503920000000001,2.1398416830000007,1.2314972610000003,6.271781921000004,2.722889557100001,10.885571694999987,0.02162278,0.039050813710000146,0.03162278,0.06837722,0.03162278,0.06837722


In [6]:
#Read the photometry
d21_raw = np.loadtxt("{}/double.21".format(folder))
nchan = len(bands)
d21 = Table()
d21['wid'] = d22['wid']
for k, bname in enumerate(bands['bname']):
    d21[bname] = d21_raw[k::nchan, 1] * u.Jy
    d21[bname+" mod"] = d21_raw[k::nchan, 2] * u.Jy
    d21[bname+" err"] = d21_raw[k::nchan, 3] * u.Jy

In [7]:
#Read in also the SDSS r-band magnitudes. 
rmag = dict()
phot = open("{}/phot.dat".format(folder))
for line in phot:
    x = line.split()
    rmag[x[0]] = float(x[-6])

In [8]:
dbase = Table.read("{}/../SED_Modeling/dbase.fits".format(folder))
coords = dict()
for wid in pran.keys():
    k = np.where(dbase['Short Name']==wid)[0][0]
    coords[wid] = SkyCoord(ra=dbase['ra'][k]*u.deg, dec=dbase['dec'][k]*u.deg)

In [9]:
coords['W0116-0505'].to_string(style='hmsdms', sep=":")

'01:16:01.411584 -05:05:04.09308'

In [10]:
#We now want to get the 6um luminosities of the unreddened best-fit quasar templates. 
seds = np.loadtxt("{}/agn_spec.dat".format(folder), skiprows=1)
agn_sed = CubicSpline(seds[:,0], seds[:,2])

In [11]:
#Now, read the SMBH masses and the Bolometric luminosities obtained from Guodong. He said the uncertainty in the BH masses is 0.4 dex and in Lbol it is 0.2 dex.
smbh = Table.read("../../Eddington_Ratios/smbh_properties.txt", format='ascii')

In [12]:
# #Calculate the 6um luminosities for each target. 
# z = d20['redshift']
# DL = cosmo.luminosity_distance(z)
# for j, vec in enumerate(['vec1', 'vec5']):
#     f_nu = d20[vec]*agn_sed(6.0) * u.Jy
#     L_nu = (4.*np.pi*DL**2)/(1+z) * f_nu
#     L6um = (c/(6*u.micron) * L_nu).to(u.erg/u.s)
#     d22['log L6um AGN{}'.format(j+1)] = np.log10(L6um.value)

In [13]:
d20.show_in_notebook()

idx,i,redshift,chi2,vec1,vec2,vec3,vec4,vec5,vec1_l1,vec1_u1,vec1_l2,vec1_u2,vec1_l3,vec1_u3,vec5_l1,vec5_u1,vec5_l2,vec5_u2,vec5_l3,vec5_u3
0,1,1.641,5.173362,0.002489157,3.060541e-06,6.749413e-06,5.600584e-06,1.939832e-05,0.0001968078745,0.00024744431250000015,0.0004161161420000001,0.0004895180169999999,0.0006254098720000004,0.0007523082859999977,6.895709620000001e-06,9.608593565000004e-06,1.1170997527000002e-05,1.9644414699999997e-05,1.4677827022000005e-05,2.745848106499999e-05
1,2,3.173,14.68472,0.004337743,0.0,7.115327e-05,0.0,3.561715e-05,0.0005934318349999999,0.0009776000689999996,0.0010480321650000002,0.002992903487000002,0.0013498469030000003,0.004476353051499991,5.071732355000001e-06,4.628885040000004e-06,1.0997175939999999e-05,1.1569021120000006e-05,1.60032523e-05,1.8669681934999924e-05
2,3,2.099307,23.1878,0.003505553,0.0,2.884517e-05,0.0,4.248803e-05,0.0005889578044999998,0.0007035407795,0.0010044477009999997,0.0016872714060000007,0.0013028514380000016,0.0028139400804998395,8.801910040000001e-06,2.184673895000002e-06,2.4899930830000006e-05,4.721848310000002e-06,4.007543934400001e-05,7.282448014999986e-06
3,4,3.122,21.3206,0.00557358,1.598678e-05,0.0,0.0,2.637247e-05,0.001299923239,0.0016764135255000001,0.0023015257020000002,0.003732098562,0.0029607749600000003,0.0060560322099999335,9.872312100000008e-07,1.5807936300000002e-06,2.389052740000003e-06,2.812407630000002e-06,5.138156880000002e-06,4.2734535599999635e-06
4,5,3.888,92.45757,0.005856387,1.830105e-05,0.0,1.161498e-05,2.257949e-05,0.0010077852515000003,0.0034439209855,0.002446652628000001,0.010316024820000002,0.005146029644850001,0.02180188291999888,1.8275981414499998e-05,1.940655313e-05,2.257949e-05,4.5114320879999995e-05,2.257949e-05,7.209723884999948e-05


In [14]:
#Calculate the 6um luminosities for each target. 
z = d20['redshift']
DL = cosmo.luminosity_distance(z)
for j, vec in enumerate(['vec1', 'vec5']):
    f_nu = d20[vec]*agn_sed(6.0) * u.Jy
    L_nu = (4.*np.pi*DL**2)/(1+z) * f_nu
    L6um = (c/(6*u.micron) * L_nu).to(u.erg/u.s)
    d22['log L6um AGN{}'.format(j+1)] = np.log10(L6um.value)
    for k in range(3):
        d22['log L6um AGN{}_l{}'.format(j+1,k+1)] = np.log10(L6um.value) - np.log10(L6um.value * (d20[vec]-d20[vec+"_l{}".format(k+1)])/d20[vec])
        d22['log L6um AGN{}_u{}'.format(j+1,k+1)] = np.log10(L6um.value * (d20[vec]+d20[vec+"_u{}".format(k+1)])/d20[vec]) - np.log10(L6um.value)

/var/folders/p7/drxzchtj4yb641v79lt0tjyh0000gn/T/ipykernel_98672/4061056990.py:10: RuntimeWarning: divide by zero encountered in log10
  d22['log L6um AGN{}_l{}'.format(j+1,k+1)] = np.log10(L6um.value) - np.log10(L6um.value * (d20[vec]-d20[vec+"_l{}".format(k+1)])/d20[vec])


In [15]:
d22.show_in_notebook()

idx,i,ebv1,ebv2,igm,wid,ebv1_l1,ebv1_u1,ebv1_l2,ebv1_u2,ebv1_l3,ebv1_u3,ebv2_l1,ebv2_u1,ebv2_l2,ebv2_u2,ebv2_l3,ebv2_u3,log L6um AGN1,log L6um AGN1_l1,log L6um AGN1_u1,log L6um AGN1_l2,log L6um AGN1_u2,log L6um AGN1_l3,log L6um AGN1_u3,log L6um AGN2,log L6um AGN2_l1,log L6um AGN2_u1,log L6um AGN2_l2,log L6um AGN2_u2,log L6um AGN2_l3,log L6um AGN2_u3
0,1,5.011872,0.03162278,1.4,W0019-1046,0.3221457944999999,0.4566297979999998,0.6405640350000006,0.9581457169999998,1.0308000000000002,1.297701,0.03162278,0.03147295,0.03162278,0.06837722,0.03162278,0.06837722,46.53290101291158,0.03577152845248577,0.041159240943677844,0.07944442819285769,0.07797083290575557,0.12566530275094578,0.1146890849776625,44.424612842233884,0.19076342186509265,0.17473740154488837,0.37250559931958804,0.3037761103841774,0.6137767633632194,0.3830085164972772
1,2,4.253576,0.0,0.4567042,W0116-0505,0.7456801134999997,0.9986133680000009,1.1274484139999998,3.579420444000001,1.5701578059999997,4.845881609499965,0.0,0.01,0.0,0.01995262,0.0,0.03162278,47.27255221705564,0.06389188470576812,0.08826748287810204,0.12010609240139303,0.22787845897870085,0.16189832664765902,0.30791396129733783,45.18694756473617,0.06671309771989797,0.053063935673989704,0.16037157428559112,0.12215557303065339,0.2590952594767373,0.18303533258381322
2,3,9.690195,0.1,1.4,W0204-0506,1.7469129999999993,1.7761171900000008,2.7960706559999995,4.4042100699999995,3.4079503769999997,7.501086569999991,0.0,0.0,0.0,0.0,0.020567180000000004,0.0,46.87462804415834,0.07988038149278509,0.07943206448942419,0.14662456119789624,0.17064709974960834,0.20180087777234945,0.25592570511467727,44.95813810167736,0.10081560417233248,0.021775644656358395,0.3830476893466326,0.04576628592015197,1.245782953597633,0.06870521683855912
3,4,7.56583,0.0,0.0,W0220+0137,1.764893538,1.9283686090000014,3.1144825030000005,3.9973169700000035,3.8130284010000013,5.890717124999968,0.0,0.0,0.0,0.0,0.0,0.01,47.36986768713766,0.11533460028402942,0.11420337960561255,0.23131373718033643,0.22261380787269047,0.329027233994573,0.31943099427343924,45.04488425498198,0.016569553954511207,0.025281713437308895,0.04123974366020633,0.044007069716279545,0.094112589818927,0.06522190701902275
4,5,3.162278,0.03162278,1.4,W0831+0140,0.6503920000000001,2.1398416830000007,1.2314972610000003,6.271781921000004,2.722889557100001,10.885571694999987,0.02162278,0.039050813710000146,0.03162278,0.06837722,0.03162278,0.06837722,47.54500994689336,0.08201325415106453,0.20086756280910834,0.23490922038001116,0.44114502391472143,0.916152886342914,0.6741952427203941,45.13109430732456,0.7198914546267545,0.2693908192184935,inf,0.4768348354116583,inf,0.6225291159784874


In [16]:
#Final table.
wids = ['W0019$-$1046', 'W0116$-$0505', 'W0204$-$0506', 'W0220+0137', 'W0831+0140']
k = list()
for wid in wids:
    k.append(np.argwhere(d22['wid']==re.sub("\$","",wid))[0][0])

Ftab = Table()
Ftab['Short WISE ID'] = wids

ra  = [None]*len(wids)
dec = [None]*len(wids)
for i, wid in enumerate(wids):
    hmsdms1 = coords[re.sub("\$","",wid)].to_string(style='hmsdms', sep=':', precision=1)
    hmsdms2 = coords[re.sub("\$","",wid)].to_string(style='hmsdms', sep=':', precision=2)
    ra[i]  = hmsdms2.split()[0]
    dec[i] = re.sub("-","$-$",hmsdms1.split()[1])
Ftab['R.A.'] = ra
Ftab['Dec.'] = dec

Ftab['r'] = 0.
Ftab['r'].info.format = '5.1f'
for i, wid in enumerate(wids):
    Ftab['r'][i] = rmag[re.sub("\$","",wid)]

Ftab['Redshift'] = d20['redshift'][k]
Ftab['Redshift'].info.format = '7.3f'

#Ftab['rmag'] = -2.5*np.log10(d21['sdssr'][k]/(3631.*u.Jy))
#Ftab['rmag'].info.format = '5.2f'

# Ftab['log L6um AGN1'] = d22['log L6um AGN1'][k]
# Ftab['log L6um AGN1'].info.format = '5.2f'

# Ftab['E(B-V) AGN1'] = d22['ebv1'][k] 
# Ftab['E(B-V) AGN1'].info.format = '5.2f'

# Ftab['log L6um AGN2'] = d22['log L6um AGN2'][k]
# Ftab['log L6um AGN2'].info.format = '5.2f'

# Ftab['E(B-V) AGN2'] = d22['ebv2'][k] 
# Ftab['E(B-V) AGN2'].info.format = '5.2f'

# We are showing things with a precision of 0.01. Errors that are formally smaller than that should be upped to 0.01 as it is most likely the gridding of the reddening parameters that is driving that. 
for cols in [('log L6um AGN1', 'log L6um AGN1'), ('E(B-V) AGN1', 'ebv1'), ('log L6um AGN2','log L6um AGN2'), ('E(B-V) AGN2','ebv2')]:
    col1, col2 = cols
    Ftab[col1] = "{:28s}".format(" ")
    d22[col2+"_u1"][d22[col2+"_u1"]<0.01] = 0.01
    if col2!="ebv2":
        d22[col2+"_l1"][d22[col2+"_l1"]<0.01] = 0.01
    else:
        d22[col2+"_l1"][(d22[col2+"_l1"]<0.01) & (d22[col2]>=0.01)] = 0.01
    for i in range(len(d22)):
        #print(col1, i, d22[col2][i], d22[col2+"_u1"][i], d22[col2+"_l1"][i])
        Ftab[col1][i] = "${:5.2f}^{{+{:4.2f}}}_{{-{:4.2f}}}$".format(d22[col2][i], d22[col2+"_u1"][i], d22[col2+"_l1"][i])

Ftab['p_ran'] = np.zeros(len(wids))
for i, wid in enumerate(wids):
    Ftab['p_ran'][i] = pran[re.sub("\$","",wid)]*1e2
Ftab['p_ran'].info.format = '5.3f'


In [17]:
#Add the SMBH properties to the table.
Ftab['log_Mbh'] = smbh['log_MBH/M_sun']
Ftab['log_Lbol'] = smbh['log_Lbol/L_sun']
Ftab['lambda_edd'] = "{:28s}".format(" ")
for i in range(len(smbh)):
    Ftab['lambda_edd'][i] = "${:5.2f}^{{+{:4.2f}}}_{{-{:4.2f}}}$".format(smbh['ledd_rat'][i], smbh['ledd_rat_err_hig'][i], smbh['ledd_rat_err_low'][i])

In [18]:
Ftab.write("targets.tex", format='latex', overwrite=True)

In [19]:
Ftab.show_in_notebook()

idx,Short WISE ID,R.A.,Dec.,r,Redshift,log L6um AGN1,E(B-V) AGN1,log L6um AGN2,E(B-V) AGN2,p_ran,log_Mbh,log_Lbol,lambda_edd
0,W0019$-$1046,00:19:26.88,$-$10:46:33.3,-99.0,1.641,$46.53^{+0.04}_{-0.04}$,$ 5.01^{+0.46}_{-0.32}$,$44.42^{+0.17}_{-0.19}$,$ 0.03^{+0.03}_{-0.03}$,10.077,9.8,13.3,$ 0.10^{+0.17}_{-0.06}$
1,W0116$-$0505,01:16:01.41,$-$05:05:04.1,-99.0,3.173,$47.27^{+0.09}_{-0.06}$,$ 4.25^{+1.00}_{-0.75}$,$45.19^{+0.05}_{-0.07}$,$ 0.00^{+0.01}_{-0.00}$,0.017,9.4,14.1,$ 1.53^{+2.75}_{-0.98}$
2,W0204$-$0506,02:04:46.13,$-$05:06:40.8,-99.0,2.099,$46.87^{+0.08}_{-0.08}$,$ 9.69^{+1.78}_{-1.75}$,$44.96^{+0.02}_{-0.10}$,$ 0.10^{+0.01}_{-0.01}$,2.729,8.8,13.7,$ 2.42^{+4.36}_{-1.55}$
3,W0220+0137,02:20:52.12,+01:37:11.6,-99.0,3.122,$47.37^{+0.11}_{-0.12}$,$ 7.57^{+1.93}_{-1.76}$,$45.04^{+0.03}_{-0.02}$,$ 0.00^{+0.01}_{-0.00}$,0.085,9.3,13.9,$ 1.21^{+2.18}_{-0.78}$
4,W0831+0140,08:31:53.25,+01:40:10.8,18.9,3.888,$47.55^{+0.20}_{-0.08}$,$ 3.16^{+2.14}_{-0.65}$,$45.13^{+0.27}_{-0.72}$,$ 0.03^{+0.04}_{-0.02}$,58.320,9.4,14.4,$ 3.04^{+5.48}_{-1.96}$
